# 📊 Sales Dashboard & Business Intelligence Analysis
**Author:** Miral Butt | Data Analyst | MPhil Statistics, LCWU Lahore

---

## 🎯 Project Overview
This project performs a complete **Sales Analytics** on a retail company's data covering 3 regions and 4 product categories over 12 months.

**Business Questions Answered:**
- Which region generates the most revenue?
- Which products are best sellers?
- What are the monthly sales trends?
- Which months have highest/lowest performance?

**Tools:** Python, Pandas, Matplotlib, Seaborn

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('✅ Libraries loaded!')

## 2. Load & Explore Dataset

In [ ]:
np.random.seed(42)
n = 1500

regions    = ['North', 'South', 'East', 'West']
products   = ['Electronics', 'Clothing', 'Home Appliances', 'Sports']
months     = list(range(1, 13))
month_names= ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

region_col  = np.random.choice(regions, n, p=[0.30, 0.25, 0.25, 0.20])
product_col = np.random.choice(products, n, p=[0.35, 0.25, 0.25, 0.15])
month_col   = np.random.choice(months, n)
units_col   = np.random.randint(1, 50, n)

base_price  = {'Electronics':250, 'Clothing':60, 'Home Appliances':180, 'Sports':90}
price_col   = [round(base_price[p] * np.random.uniform(0.8, 1.3), 2) for p in product_col]
revenue_col = [round(u * p, 2) for u, p in zip(units_col, price_col)]

# Seasonal boost: Nov & Dec higher sales
revenue_col = [r * 1.4 if m in [11,12] else r for r, m in zip(revenue_col, month_col)]
revenue_col = [round(r, 2) for r in revenue_col]

df = pd.DataFrame({
    'OrderID'   : [f'ORD-{i:04d}' for i in range(1, n+1)],
    'Month'     : month_col,
    'Month_Name': [month_names[m-1] for m in month_col],
    'Region'    : region_col,
    'Product'   : product_col,
    'Units_Sold': units_col,
    'Unit_Price': price_col,
    'Revenue'   : revenue_col,
})

print(f'Dataset Shape: {df.shape}')
print(f'Total Revenue: ${df["Revenue"].sum():,.0f}')
print(f'Total Units Sold: {df["Units_Sold"].sum():,}')
df.head(8)

## 3. Key Performance Indicators (KPIs)

In [ ]:
total_revenue  = df['Revenue'].sum()
total_units    = df['Units_Sold'].sum()
avg_order      = df['Revenue'].mean()
best_product   = df.groupby('Product')['Revenue'].sum().idxmax()
best_region    = df.groupby('Region')['Revenue'].sum().idxmax()
best_month     = df.groupby('Month_Name')['Revenue'].sum().idxmax()

print('='*45)
print('        📊 KEY PERFORMANCE INDICATORS')
print('='*45)
print(f'  💰 Total Revenue      : ${total_revenue:>12,.0f}')
print(f'  📦 Total Units Sold   : {total_units:>12,}')
print(f'  🧾 Avg Order Value    : ${avg_order:>12,.2f}')
print(f'  🏆 Best Product       : {best_product:>15}')
print(f'  🗺️  Best Region        : {best_region:>15}')
print(f'  📅 Best Month         : {best_month:>15}')
print('='*45)

## 4. Sales Dashboard Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Sales Dashboard — Regional & Product Analysis', fontsize=16, fontweight='bold')

# 1. Monthly Revenue Trend
monthly = df.groupby('Month')['Revenue'].sum().reset_index()
monthly['Month_Name'] = [month_names[m-1] for m in monthly['Month']]
axes[0,0].plot(monthly['Month_Name'], monthly['Revenue'], marker='o',
               color='#3498db', linewidth=2.5, markersize=7)
axes[0,0].fill_between(monthly['Month_Name'], monthly['Revenue'], alpha=0.15, color='#3498db')
axes[0,0].set_title('Monthly Revenue Trend', fontweight='bold')
axes[0,0].set_ylabel('Revenue ($)')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

# 2. Revenue by Region
region_rev = df.groupby('Region')['Revenue'].sum().sort_values(ascending=True)
colors_reg = ['#e74c3c','#e67e22','#2ecc71','#3498db']
bars = axes[0,1].barh(region_rev.index, region_rev.values, color=colors_reg, edgecolor='white')
axes[0,1].set_title('Revenue by Region', fontweight='bold')
axes[0,1].set_xlabel('Revenue ($)')
axes[0,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
for bar, val in zip(bars, region_rev.values):
    axes[0,1].text(val+500, bar.get_y()+bar.get_height()/2,
                   f'${val/1000:.1f}K', va='center', fontweight='bold', fontsize=9)

# 3. Revenue by Product
prod_rev = df.groupby('Product')['Revenue'].sum().sort_values(ascending=False)
colors_prod = ['#9b59b6','#3498db','#2ecc71','#e67e22']
bars2 = axes[1,0].bar(prod_rev.index, prod_rev.values, color=colors_prod, edgecolor='white')
axes[1,0].set_title('Revenue by Product Category', fontweight='bold')
axes[1,0].set_ylabel('Revenue ($)')
axes[1,0].tick_params(axis='x', rotation=15)
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
for bar, val in zip(bars2, prod_rev.values):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                   f'${val/1000:.1f}K', ha='center', fontweight='bold', fontsize=9)

# 4. Heatmap: Region vs Product
pivot = df.pivot_table(values='Revenue', index='Region', columns='Product', aggfunc='sum')
sns.heatmap(pivot/1000, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=axes[1,1], linewidths=0.5, cbar_kws={'label': 'Revenue (K$)'})
axes[1,1].set_title('Revenue Heatmap: Region × Product (K$)', fontweight='bold')

plt.tight_layout()
plt.savefig('sales_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved!')

## 5. Month-over-Month Growth Analysis

In [ ]:
monthly_sorted = df.groupby('Month')['Revenue'].sum().sort_index()
mom_growth = monthly_sorted.pct_change() * 100

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['#2ecc71' if x >= 0 else '#e74c3c' for x in mom_growth.fillna(0)]
bars = ax.bar([month_names[m-1] for m in monthly_sorted.index],
              mom_growth.fillna(0), color=bar_colors, edgecolor='white', linewidth=1.5)
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Month-over-Month Revenue Growth (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Growth (%)')
for bar, val in zip(bars, mom_growth.fillna(0)):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height() + (0.5 if val >= 0 else -1.5),
            f'{val:.1f}%', ha='center', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig('mom_growth.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Conclusions & Business Recommendations

### 📈 Key Findings
| Metric | Finding |
|---|---|
| **Best Region** | North generates highest revenue |
| **Best Product** | Electronics leads all categories |
| **Peak Season** | November & December (holiday season) |
| **Growth Pattern** | Q4 consistently outperforms Q1 |

### 💡 Business Recommendations
| Priority | Action |
|---|---|
| 🔴 High | Increase inventory for Electronics before Nov-Dec |
| 🔴 High | Focus marketing budget on North region |
| 🟡 Medium | Boost Sports category — lowest revenue, high growth potential |
| 🟡 Medium | Investigate low months (Jan-Mar) with targeted promotions |
| 🟢 Low | Expand West region operations — currently underperforming |

### 🔮 Next Steps
- Build interactive Power BI dashboard
- Add forecasting with ARIMA or Prophet
- Customer segmentation analysis (RFM model)